## Notes
- do we need a seed in initializing the EA? default should be a random seed
- do we need to force typeguard on the initialization? User-unfriendly and should be casted in our side.
-    1256     logger.error(
   1257         "--include-dashboard is not supported when minimal ray is used. "
   1258         "Download ray[default] to use the dashboard."
   1259     )
   we don't need to deploy the default ray if we don't care to have the dashboard

In [ ]:
from starbase_gp import EA
import numpy as np
# Initialize the evolutionary algorithm
ea = EA(seed=np.uint16(42), 
        pop_size=np.uint16(50),
        uni_cnt_max=np.uint16(10),
        uni_cnt_min=np.uint16(2),
        cores=4)


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x1205ed610>>
Traceback (most recent call last):
  File "/Users/matsumoton/miniconda3/envs/starbase/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


In [ ]:
#/Users/matsumoton/Library/CloudStorage/OneDrive-Cedars-SinaiHealthSystem/StarBASE-GP/Benchmarking
file_path = "/Users/matsumoton/Library/CloudStorage/OneDrive-Cedars-SinaiHealthSystem/StarBASE-GP/Benchmarking/top10k_pruned_BMIres.csv"

# Load data
ea.data_loader(file_path, target_label="y", split=0.5)

# Initialize hubs
ea.initialize_hubs(bin_size=100)

Loading data...
Path: /Users/matsumoton/Library/CloudStorage/OneDrive-Cedars-SinaiHealthSystem/StarBASE-GP/Benchmarking/top10k_pruned_BMIres.csv
Data loaded successfully.
Data shape: (5566, 10001)
X_data.shape: (5566, 10000)
y_data.shape: (5566,)

Genotype data:        1.281788173  1.282043175  1.282025017  1.281823764  1.281881111  \
0             0.5          0.5          0.5          0.5          0.5   
1             1.0          1.0          1.0          1.0          1.0   
2             0.0          0.0          0.0          0.0          0.0   
3             0.5          0.5          0.5          0.5          0.5   
4             0.0          0.5          1.0          1.0          1.0   
...           ...          ...          ...          ...          ...   
5561          1.0          1.0          1.0          1.0          1.0   
5562          1.0          1.0          1.0          1.0          1.0   
5563          0.5          0.5          0.5          0.5          0.5   
5564  

In [ ]:

# Run evolution
ea.evolve(gens=50)

# Run post analysis
ea.post_analysis()

Initializing population...


AssertionError: 

In [ ]:
# to run: clear; python runner.py --seed 0 --pop_size 50 --uni_cnt_max 100 --uni_cnt_min 50 --cores 4 --mut_ran_p 0.5 --mut_smt_p 0.5 --smt_in_in_p 0.45 --smt_in_out_p 0.45 --smt_out_out_p 0.10 --num_add_interactions 10 --num_del_interactions 10 --data_dir /Users/matsumoton/Library/CloudStorage/OneDrive-Cedars-SinaiHealthSystem/StarBASE-GP/Benchmarking/top10k_pruned_BMIres.csv  --save_directory ./ --bin_size 10 --gens 1

import argparse
import numpy as np
import ray
from starbase_gp import EA
import time
import os
# os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
# os.environ["RAY_DISABLE_DASHBOARD"] = "1"
ray.shutdown()

def main(args):
    # set experiment configurations using arguments from SLURM array job
    ea_config = {
        'seed': np.uint16(args.seed),
        'pop_size': np.uint16(args.pop_size),
        'uni_cnt_max': np.uint16(args.uni_cnt_max),
        'uni_cnt_min': np.uint16(args.uni_cnt_min),
        'cores': args.cores,
        'mut_selector_p': np.float64(1.0),  # This remains constant
        'mut_regressor_p': np.float64(.5),  # This remains constant
        'mut_ran_p': np.float64(args.mut_ran_p),
        'mut_smt_p': np.float64(args.mut_smt_p),
        'smt_in_in_p': np.float64(args.smt_in_in_p),
        'smt_in_out_p': np.float64(args.smt_in_out_p),
        'smt_out_out_p': np.float64(args.smt_out_out_p),
        'mut_prob': np.float64(.5),         # Assuming constant
        'cross_prob': np.float64(.5),       # Assuming constant
        'num_add_interactions': np.uint16(args.num_add_interactions),
        'num_del_interactions': np.uint16(args.num_del_interactions),
        'save_directory': args.save_directory,
        'debug' : args.debug
    }

    for i in range(1):
        # create a new directory for each run
        ea_config['save_directory'] = f"{args.save_directory}/Run_{i}/"

        # make the directory
        os.makedirs(ea_config['save_directory'], exist_ok=True)

        try:
            # update seed
            ea_config['seed'] = np.uint16(i)

            ea = EA(**ea_config)

            total_time = time.time()

            # Update to use the data_dir passed as argument
            start_time = time.time()
            ea.data_loader(args.data_dir)
            print(f"Data loaded in {(time.time() - start_time) / 60} mins")

            start_time = time.time()
            ea.initialize_hubs(args.bin_size)
            print(f"Hubs done in {(time.time() - start_time) / 60} mins")

            start_time = time.time()
            ea.evolve(args.gens)
            print(f"Evolution done in {(time.time() - start_time) / 60 / 60} hours")

            start_time = time.time()
            ea.post_analysis()
            print(f"Post analysis done in {(time.time() - start_time) / 60 / 60} hours")

            ea.hubs.save_hubs("snp_hub_"+time.time()+".csv")

            print(f"Time taken: {(time.time() - total_time) / 60 / 60} hours")

        except Exception as e:
            # create a log.txt file to store the error
            with open(f"{args.save_directory}/log_{i}.txt", "a") as f:
                f.write(f"Error: {e}\n")

        ray.shutdown()



In [ ]:
import pandas as pd
#read in csv '/Users/matsumoton/Downloads/rat_data_starbase_maf01_M05_modeimpute.csv'
csv_path = '/Users/matsumoton/Downloads/rat_data_starbase_maf01_M05_modeimpute.csv'
data = pd.read_csv(csv_path)

In [ ]:
#save csv as feather file
data.to_feather('/Users/matsumoton/Downloads/rat_data_starbase_maf01_M05_modeimpute.feather')

In [ ]:
import pandas as pd
#load in feather file
data = pd.read_feather('/Users/matsumoton/Downloads/rat_data_starbase_maf01_M05_modeimpute.feather')


In [ ]:
data.head()

,1.55365,1.275972,1.475912,1.759319,1.835524,1.923940,1.1131973,1.1132321,1.1142175,1.1150700,...,20.55561229,20.55575749,20.55622582,20.55625042,20.55637803,20.55744179,20.55944461,20.55991426,20.55999627,y
0,2,1,1,0,1,2,1,1,2,1,...,2,2,2,2,2,2,2,2,2,-0.033982
1,2,1,1,1,1,2,1,1,2,1,...,2,2,2,2,2,2,2,2,2,-0.297964
2,2,1,1,1,1,1,1,1,1,1,...,2,2,2,2,2,2,2,2,2,1.134162
3,2,1,1,1,1,2,1,1,2,1,...,1,1,1,1,1,1,1,1,1,-1.248074
4,2,1,1,0,1,2,1,1,2,0,...,2,2,2,2,2,2,2,2,2,0.953451


In [ ]:

import argparse
import numpy as np
import ray
from starbase_gp import EA
import time
import os
# os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
# os.environ["RAY_DISABLE_DASHBOARD"] = "1"
ray.shutdown()
ea_config = {'seed': np.uint16(42), 'pop_size': np.uint16(100), 'uni_cnt_max': np.uint16(10), 'uni_cnt_min': np.uint16(2), 'cores': 4, 'mut_selector_p': np.float64(1.0), 'mut_regressor_p': np.float64(0.5), 'mut_ran_p': np.float64(0.5), 'mut_smt_p': np.float64(0.5), 'smt_in_in_p': np.float64(0.45), 'smt_in_out_p': np.float64(0.45), 'smt_out_out_p': np.float64(0.1), 'mut_prob': np.float64(0.5), 'cross_prob': np.float64(0.5), 'num_add_interactions': np.uint16(10), 'num_del_interactions': np.uint16(10), 'save_directory': './', 'debug': False}
# create a new directory for each run
i=0
ea_config['save_directory'] = f"{str(ea_config['save_directory'])}/Run_{str(i)}/"

# make the directory
os.makedirs(ea_config['save_directory'], exist_ok=True)
#"/Users/matsumoton/Library/CloudStorage/OneDrive-Cedars-SinaiHealthSystem/StarBASE-GP/Benchmarking/top10k_pruned_BMIres.csv"
# args ={'data_dir': '/Users/matsumoton/Library/CloudStorage/OneDrive-Cedars-SinaiHealthSystem/StarBASE-GP/Benchmarking/top10k_pruned_BMIres.csv', 'bin_size': 10, 'gens': 1}
args ={'data_dir': '/Users/matsumoton/Downloads/rat_data_starbase_maf01_M05_modeimpute.feather', 'bin_size': 10, 'gens': 1}

# update seed
ea_config['seed'] = np.uint16(i)

ea = EA(**ea_config)

total_time = time.time()

# Update to use the data_dir passed as argument
start_time = time.time()
ea.data_loader(args['data_dir'])
print(f"Data loaded in {(time.time() - start_time) / 60} mins")

start_time = time.time()
ea.initialize_hubs(args['bin_size'])
print(f"Hubs done in {(time.time() - start_time) / 60} mins")

start_time = time.time()
ea.evolve(args['gens'])
print(f"Evolution done in {(time.time() - start_time) / 60 / 60} hours")

start_time = time.time()
ea.post_analysis()
print(f"Post analysis done in {(time.time() - start_time) / 60 / 60} hours")

ea.hubs.save_hubs("snp_hub_"+str(time.time())+".csv")

print(f"Time taken: {(time.time() - total_time) / 60 / 60} hours")

ray.shutdown()

#matrix optimized with handmade linear regression - Unseen snps evaluated in 0.15875715017318726 mins
#matrix optimized with sklearn LinearRegression 
#non-optimized? - Unseen snps evaluated in 0.1711354931195577 mins



2024-12-30 10:16:56,620	INFO worker.py:1812 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 



Loading data...
Path: /Users/matsumoton/Downloads/rat_data_starbase_maf01_M05_modeimpute.feather
Data loaded successfully.
Data shape: (3166, 103815)
X_data.shape: (3166, 103814)
y_data.shape: (3166,)


/Users/matsumoton/Git/StarBASE/starbase_gp/evolver.py:437: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  all_x = all_x.applymap(lambda x: 0.5 if x == 1 else (1 if x == 2 else x))


Genotype data:     1.55365  1.275972  1.475912  1.759319  1.835524  1.923940  1.1131973  \
0      1.0       0.5       0.5       0.0       0.5       1.0        0.5   
1      1.0       0.5       0.5       0.5       0.5       1.0        0.5   
2      1.0       0.5       0.5       0.5       0.5       0.5        0.5   
3      1.0       0.5       0.5       0.5       0.5       1.0        0.5   
4      1.0       0.5       0.5       0.0       0.5       1.0        0.5   

   1.1132321  1.1142175  1.1150700  ...  20.55274746  20.55561229  \
0        0.5        1.0        0.5  ...          1.0          1.0   
1        0.5        1.0        0.5  ...          1.0          1.0   
2        0.5        0.5        0.5  ...          1.0          1.0   
3        0.5        1.0        0.5  ...          0.5          0.5   
4        0.5        1.0        0.0  ...          1.0          1.0   

   20.55575749  20.55622582  20.55625042  20.55637803  20.55744179  \
0          1.0          1.0          1.0        

(raylet) Spilled 2510 MiB, 4 objects, write throughput 2105 MiB/s. Set RAY_verbose_spill_logs=0 to disable this message.


(ray_uni_eval_optimized pid=95127) All encodings train shape:  (1583, 7)
(ray_uni_eval_optimized pid=95127) All encodings val shape:  (1583, 7)
Unseen snps evaluated in 0.1571741302808126 mins



AssertionError: 

In [ ]:
#python runner.py --seed 0 --pop_size 50 --uni_cnt_max 100 --uni_cnt_min 50 --cores 1 --mut_ran_p 0.5 --mut_smt_p 0.5 --smt_in_in_p 0.45 --smt_in_out_p 0.45 --smt_out_out_p 0.10 --num_add_interactions 10 --num_del_interactions 10 --data_dir /Users/matsumoton/Library/CloudStorage/OneDrive-Cedars-SinaiHealthSystem/StarBASE-GP/Benchmarking/top10k_pruned_BMIres.csv  --save_directory ./ --bin_size 10 --gens 1
#runs in 27 minutes with 1 core
#runs in 8 minutes with 4 cores
# runs in 3.5 minutes with 8 cores


args = argparse.Namespace(seed=0, pop_size=100, uni_cnt_max=100, uni_cnt_min=50, cores=8, mut_ran_p=0.5, mut_smt_p=0.5, smt_in_in_p=0.45, smt_in_out_p=0.45, smt_out_out_p=0.10, num_add_interactions=10, num_del_interactions=10, data_dir="/Users/matsumoton/Library/CloudStorage/OneDrive-Cedars-SinaiHealthSystem/StarBASE-GP/Benchmarking/top10k_pruned_BMIres.csv", save_directory="./", bin_size=10, gens=1, debug=False)
main(args)

2024-12-18 01:26:58,691	INFO worker.py:1812 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 



Loading data...
Path: /Users/matsumoton/Library/CloudStorage/OneDrive-Cedars-SinaiHealthSystem/StarBASE-GP/Benchmarking/top10k_pruned_BMIres.csv
Data loaded successfully.
Data shape: (5566, 10001)
X_data.shape: (5566, 10000)
y_data.shape: (5566,)

Genotype data:        1.281788173  1.282043175  1.282025017  1.281823764  1.281881111  \
0             0.5          0.5          0.5          0.5          0.5   
1             1.0          1.0          1.0          1.0          1.0   
2             0.0          0.0          0.0          0.0          0.0   
3             0.5          0.5          0.5          0.5          0.5   
4             0.0          0.5          1.0          1.0          1.0   
...           ...          ...          ...          ...          ...   
5561          1.0          1.0          1.0          1.0          1.0   
5562          1.0          1.0          1.0          1.0          1.0   
5563          0.5          0.5          0.5          0.5          0.5   
5564 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from concurrent.futures import ThreadPoolExecutor
from functools import partial

# Define constants
ENCODINGS = ['dominant', 'recessive', 'heterosis', 'underdominant', 
             'subadditive', 'superadditive', 'pager']

def _apply_basic_mapping(snp_values, mapping):
    """Vectorized mapping application"""
    result = np.zeros_like(snp_values, dtype=np.float32)
    for k, v in mapping.items():
        result[snp_values == k] = v
    return result

def _compute_pager_encoding(snp_values, y):
    """Compute PAGER encoding for a single column"""
    df = pd.DataFrame({'snp': snp_values, 'Phenotype': y})
    geno_agg = df.groupby('snp').agg(mean_phenotype=('Phenotype', 'mean')).reset_index()
    anchor_mean = geno_agg.loc[geno_agg['snp'].idxmin(), 'mean_phenotype']
    geno_agg['rel_dist'] = geno_agg['mean_phenotype'] - anchor_mean
    
    scaler = MinMaxScaler()
    geno_agg['normalized_rel_dist'] = scaler.fit_transform(
        geno_agg['rel_dist'].values.reshape(-1, 1)
    ).ravel()
    
    mapping = dict(zip(geno_agg['snp'], geno_agg['normalized_rel_dist']))
    result = np.array([mapping.get(val, 0.5) for val in snp_values], dtype=np.float32)
    return result

def map_snp_values(X, snp_pos, encoding_type='all', y=None):
    """
    Map SNP values based on the encoding type, with optimized 'all' option.
    
    Args:
        X (pd.DataFrame or np.ndarray): Input data
        snp_pos (int): Position of the SNP column
        encoding_type (str): Encoding type ('all' or specific encoding)
        y (np.ndarray, optional): Phenotype values for PAGER encoding
    
    Returns:
        np.ndarray: Mapped values, shape (n_samples, 1) or (n_samples, n_encodings) if encoding_type='all'
    """
    # Extract SNP values
    snp_values = X.iloc[:, snp_pos].values if isinstance(X, pd.DataFrame) else X[:, snp_pos]
    
    # Define basic mappings
    mappings = {
        'dominant': {0: 0, 0.5: 1, 1: 1},
        'recessive': {0: 0, 0.5: 0, 1: 1},
        'heterosis': {0: 0, 0.5: 1, 1: 0},
        'underdominant': {0: 0.5, 0.5: 0, 1: 1},
        'subadditive': {0: 0, 0.5: 0.25, 1: 1},
        'superadditive': {0: 0, 0.5: 0.75, 1: 1}
    }
    
    if encoding_type == 'all':
        # Initialize result matrix
        n_samples = len(snp_values)
        result = np.zeros((n_samples, len(ENCODINGS)), dtype=np.float32)
        
        # Apply basic mappings in parallel
        for i, enc in enumerate(ENCODINGS[:-1]):  # Exclude PAGER
            result[:, i] = _apply_basic_mapping(snp_values, mappings[enc])
        
        # Compute PAGER encoding if y is provided
        if y is not None:
            result[:, -1] = _compute_pager_encoding(snp_values, y)
        else:
            result[:, -1] = 0.5  # Default value if no phenotype data
            
        return result
    
    elif encoding_type == 'pager':
        if y is None:
            raise ValueError("Phenotype values (y) must be provided for PAGER encoding.")
        return _compute_pager_encoding(snp_values, y).reshape(-1, 1)
    
    elif encoding_type in mappings:
        return _apply_basic_mapping(snp_values, mappings[encoding_type]).reshape(-1, 1)
    
    else:
        raise ValueError(f"Unsupported encoding type. Must be one of {['all'] + ENCODINGS}")

snp_size = 10000000
# Example usage
X = pd.DataFrame({
    'snp1': list(np.random.choice([0, 0.5, 1], snp_size)),
    # 'snp2': [1, 0.5, 0, 1, 0.5],
    #random array of 10000 values either 0, 0.5, or 1
    'snp3' : list(np.random.choice([0, 0.5, 1], snp_size))
})
y = np.random.rand(snp_size)

# Get all encodings
all_encodings = map_snp_values(X, snp_pos=0, encoding_type='all', y=y)
print("All encodings matrix shape:", all_encodings.shape)
print("All encodings:\n", all_encodings)




All encodings matrix shape: (10000000, 7)
All encodings:
 [[1.         0.         1.         ... 0.25       0.75       1.        ]
 [1.         0.         1.         ... 0.25       0.75       1.        ]
 [1.         0.         1.         ... 0.25       0.75       1.        ]
 ...
 [1.         0.         1.         ... 0.25       0.75       1.        ]
 [0.         0.         0.         ... 0.         0.         0.03343794]
 [1.         0.         1.         ... 0.25       0.75       1.        ]]


In [26]:
# run linear regression per column on the encoded data and compute the r^2 value per column
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def compute_r2(X, y):
    """
    Compute R^2 values for each column in X.
    
    Args:
        X (np.ndarray): Input data, shape (n_samples, n_features)
        y (np.ndarray): Target values, shape (n_samples,)
    
    Returns:
        np.ndarray: R^2 values for each column, shape (n_features,)
    """
    r2_values = np.zeros(X.shape[1])
    for i in range(X.shape[1]):
        reg = LinearRegression().fit(X[:, i].reshape(-1, 1), y)
        r2_values[i] = reg.score(X[:, i].reshape(-1, 1), y)
    return r2_values

# Example usage
r2_values = compute_r2(all_encodings, y)
print("R^2 values for each encoding:")
for i, enc in enumerate(ENCODINGS):
    print(f"{enc}: {r2_values[i]:.16f}")




R^2 values for each encoding:
dominant: 0.0000000995325861
recessive: 0.0000001219646416
heterosis: 0.0000004418196351
underdominant: 0.0000003427363190
subadditive: 0.0000000412466719
superadditive: 0.0000000274421605
pager: 0.0000004422027777


In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

def compute_r2_optimized(X_train, X_test, y_train, y_test):
    """
    Vectorized computation of R² values for each column using train/test split.
    
    Args:
        X_train (np.ndarray): Training data, shape (n_train_samples, n_features)
        X_test (np.ndarray): Test data, shape (n_test_samples, n_features) 
        y_train (np.ndarray): Training target values, shape (n_train_samples,)
        y_test (np.ndarray): Test target values, shape (n_test_samples,)
    
    Returns:
        np.ndarray: R² values for each column on test data, shape (n_features,)
    """
    # Ensure inputs are numpy arrays
    X_train = np.asarray(X_train)
    X_test = np.asarray(X_test)
    y_train = np.asarray(y_train)
    y_test = np.asarray(y_test)

    # Preallocate array for R² scores
    r2_scores = np.zeros(X_train.shape[1])

    # Compute R² for each feature column
    for i in range(X_train.shape[1]):
        # Reshape for single feature regression
        X_train_col = X_train[:, i].reshape(-1, 1)
        X_test_col = X_test[:, i].reshape(-1, 1)

        # Fit linear regression model
        model = LinearRegression()
        model.fit(X_train_col, y_train)

        # Predict on test data
        y_pred = model.predict(X_test_col)

        # Compute R² score
        r2_scores[i] = r2_score(y_test, y_pred)

    return r2_scores

# Example usage
r2_values = compute_r2_optimized(X_train, X_test, y_train, y_test)
print("R^2 values for each encoding:")
for i, enc in enumerate(ENCODINGS):
    print(f"{enc}: {r2_values[i]:.16f}")

R^2 values for each encoding:
dominant: 0.0000000995325861
recessive: 0.0000001219646416
heterosis: 0.0000004418196351
underdominant: 0.0000003427363190
subadditive: 0.0000000412466719
superadditive: 0.0000000274421605
pager: 0.0000004422027777


In [ ]:
import numpy as np

def compute_r2_optimized_matrix(X_train, X_test, y_train, y_test):
    """
    Optimized computation of R² values for each column using matrix operations.
    
    Args:
        X_train (np.ndarray): Training data, shape (n_train_samples, n_features)
        X_test (np.ndarray): Test data, shape (n_test_samples, n_features) 
        y_train (np.ndarray): Training target values, shape (n_train_samples,)
        y_test (np.ndarray): Test target values, shape (n_test_samples,)
    
    Returns:
        np.ndarray: R² values for each column on test data, shape (n_features,)
    """
    # Ensure inputs are numpy arrays
    X_train = np.asarray(X_train)
    X_test = np.asarray(X_test)
    y_train = np.asarray(y_train)
    y_test = np.asarray(y_test)

    # Center y_train and y_test for proper R² computation
    y_train_mean = y_train.mean()
    y_test_mean = y_test.mean()
    y_train_centered = y_train - y_train_mean
    y_test_centered = y_test - y_test_mean

    # Compute coefficients for each column (closed-form solution)
    # Add a column of ones to X for the intercept term
    intercept = np.ones((X_train.shape[0], 1))
    X_train_intercept = np.hstack([intercept, X_train])

    # (X^T X)^(-1) X^T y
    XtX_inv = np.linalg.pinv(X_train_intercept.T @ X_train_intercept)  # Pseudo-inverse for stability
    coefficients = XtX_inv @ X_train_intercept.T @ y_train_centered

    # Split intercept and slopes
    intercepts = coefficients[0]
    slopes = coefficients[1:]

    # Predict for each feature on test data
    intercept_test = np.ones((X_test.shape[0], 1))
    X_test_intercept = np.hstack([intercept_test, X_test])
    y_pred = X_test_intercept @ coefficients

    # Compute R² scores for each feature column
    rss = np.sum((y_test_centered - y_pred) ** 2, axis=0)  # Residual sum of squares
    tss = np.sum(y_test_centered ** 2)  # Total sum of squares
    r2_scores = 1 - (rss / tss)

    return r2_scores

# Example usage
r2_values = compute_r2_optimized_matrix(X_train.iloc[:, 0], X_test.iloc[:, 0], y_train, y_test)
print("R^2 values for each encoding:")
for i, enc in enumerate(ENCODINGS):
    print(f"{enc}: {r2_values[i]:.16f}")
    

: 

: 

In [15]:
import numpy as np
from sklearn.metrics import r2_score

def compute_r2_optimized(X_train, X_test, y_train, y_test):
    """
    Vectorized computation of R² values for each column using train/test split.
    
    Args:
        X_train (np.ndarray): Training data, shape (n_train_samples, n_features)
        X_test (np.ndarray): Test data, shape (n_test_samples, n_features) 
        y_train (np.ndarray): Training target values, shape (n_train_samples,)
        y_test (np.ndarray): Test target values, shape (n_test_samples,)
    
    Returns:
        np.ndarray: R² values for each column on test data, shape (n_features,)
    """
    # Input validation
    if X_train.shape[1] != X_test.shape[1]:
        raise ValueError("Training and test data must have same number of features")
    
    # Compute means from training data
    X_train_mean = X_train.mean(axis=0)
    y_train_mean = y_train.mean()
    
    # Center test data using training means
    X_test_centered = X_test - X_train_mean
    y_test_centered = y_test - y_train_mean
    
    # Compute coefficients using training data
    X_train_centered = X_train - X_train_mean
    y_train_centered = y_train - y_train_mean
    beta = np.sum(X_train_centered * y_train_centered.reshape(-1, 1), axis=0) / np.sum(X_train_centered**2, axis=0)
    
    # Compute predictions on test data
    y_pred = X_test_centered * beta + y_train_mean
    
    # Compute R² for each feature using test data
    r2_values = np.zeros(X_test.shape[1])
    for i in range(X_test.shape[1]):
        r2_values[i] = r2_score(y_test, y_pred[:, i])
    
    return r2_values

# Example usage
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(all_encodings, y, test_size=0.5, random_state=42)
r2_values = compute_r2_optimized(X_train, X_test, y_train, y_test)

print("Test R² values for each encoding:")
for i, enc in enumerate(ENCODINGS):
    print(f"{enc}: {r2_values[i]:.4f}")

Test R² values for each encoding:
dominant: -0.0000
recessive: -0.0000
heterosis: -0.0000
underdominant: -0.0000
subadditive: -0.0000
superadditive: -0.0000
pager: -0.0000


In [10]:
snp_size = 5000000
# Example usage
X = pd.DataFrame({
    'snp1': list(np.random.choice([0, 0.5, 1], snp_size)),
    # 'snp2': [1, 0.5, 0, 1, 0.5],
    #random array of 10000 values either 0, 0.5, or 1
    'snp3' : list(np.random.choice([0, 0.5, 1], snp_size))
})
y = np.random.rand(snp_size)

# Get all encodings
all_encodings = map_snp_values(X, snp_pos=0, encoding_type='all', y=y)
print("All encodings matrix shape:", all_encodings.shape)
print("All encodings:\n", all_encodings)

# Example usage
r2_values = compute_r2_optimized(all_encodings, y)
print("R^2 values for each encoding:")
#get best encoding
best_encoding = ENCODINGS[np.argmax(r2_values)]
print(f"Best encoding: {best_encoding} with R² = {r2_values.max():.16f}")
for i, enc in enumerate(ENCODINGS):
    print(f"{enc}: {r2_values[i]:.16f}")



All encodings matrix shape: (5000000, 7)
All encodings:
 [[1.        1.        0.        ... 1.        1.        0.       ]
 [1.        1.        0.        ... 1.        1.        0.       ]
 [1.        0.        1.        ... 0.25      0.75      0.3499395]
 ...
 [1.        0.        1.        ... 0.25      0.75      0.3499395]
 [0.        0.        0.        ... 0.        0.        1.       ]
 [0.        0.        0.        ... 0.        0.        1.       ]]


TypeError: compute_r2_optimized() missing 2 required positional arguments: 'y_train' and 'y_test'

In [6]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score

# Define constants
ENCODINGS = ['dominant', 'recessive', 'heterosis', 'underdominant', 
             'subadditive', 'superadditive', 'pager']

def _apply_basic_mapping(snp_values, mapping):
    """Vectorized mapping application"""
    result = np.zeros_like(snp_values, dtype=np.float32)
    for k, v in mapping.items():
        result[snp_values == k] = v
    return result

def _compute_pager_encoding(snp_values, y):
    """Compute PAGER encoding for a single column"""
    df = pd.DataFrame({'snp': snp_values, 'Phenotype': y})
    geno_agg = df.groupby('snp').agg(mean_phenotype=('Phenotype', 'mean')).reset_index()
    anchor_mean = geno_agg.loc[geno_agg['snp'].idxmin(), 'mean_phenotype']
    geno_agg['rel_dist'] = geno_agg['mean_phenotype'] - anchor_mean
    
    scaler = MinMaxScaler()
    geno_agg['normalized_rel_dist'] = scaler.fit_transform(
        geno_agg['rel_dist'].values.reshape(-1, 1)
    ).ravel()
    
    mapping = dict(zip(geno_agg['snp'], geno_agg['normalized_rel_dist']))
    result = np.array([mapping.get(val, 0.5) for val in snp_values], dtype=np.float32)
    return result

def map_snp_values(X, snp_pos, encoding_type='all', y=None):
    """
    Map SNP values based on the encoding type, with optimized 'all' option.
    
    Args:
        X (pd.DataFrame or np.ndarray): Input data
        snp_pos (int): Position of the SNP column
        encoding_type (str): Encoding type ('all' or specific encoding)
        y (np.ndarray, optional): Phenotype values for PAGER encoding
    
    Returns:
        np.ndarray: Mapped values, shape (n_samples, 1) or (n_samples, n_encodings) if encoding_type='all'
    """
    # Extract SNP values
    snp_values = X.iloc[:, snp_pos].values if isinstance(X, pd.DataFrame) else X[:, snp_pos]
    
    # Define basic mappings
    mappings = {
        'dominant': {0: 0, 0.5: 1, 1: 1},
        'recessive': {0: 0, 0.5: 0, 1: 1},
        'heterosis': {0: 0, 0.5: 1, 1: 0},
        'underdominant': {0: 0.5, 0.5: 0, 1: 1},
        'subadditive': {0: 0, 0.5: 0.25, 1: 1},
        'superadditive': {0: 0, 0.5: 0.75, 1: 1}
    }
    
    if encoding_type == 'all':
        # Initialize result matrix
        n_samples = len(snp_values)
        result = np.zeros((n_samples, len(ENCODINGS)), dtype=np.float32)
        
        # Apply basic mappings in parallel
        for i, enc in enumerate(ENCODINGS[:-1]):  # Exclude PAGER
            result[:, i] = _apply_basic_mapping(snp_values, mappings[enc])
        
        # Compute PAGER encoding if y is provided
        if y is not None:
            result[:, -1] = _compute_pager_encoding(snp_values, y)
        else:
            result[:, -1] = 0.5  # Default value if no phenotype data
            
        return result
    
    elif encoding_type == 'pager':
        if y is None:
            raise ValueError("Phenotype values (y) must be provided for PAGER encoding.")
        return _compute_pager_encoding(snp_values, y).reshape(-1, 1)
    
    elif encoding_type in mappings:
        return _apply_basic_mapping(snp_values, mappings[encoding_type]).reshape(-1, 1)
    
    else:
        raise ValueError(f"Unsupported encoding type. Must be one of {['all'] + ENCODINGS}")


def compute_r2_optimized(X_train, X_test, y_train, y_test):
    """
    Vectorized computation of R² values for each column using train/test split.
    
    Args:
        X_train (np.ndarray): Training data, shape (n_train_samples, n_features)
        X_test (np.ndarray): Test data, shape (n_test_samples, n_features) 
        y_train (np.ndarray): Training target values, shape (n_train_samples,)
        y_test (np.ndarray): Test target values, shape (n_test_samples,)
    
    Returns:
        np.ndarray: R² values for each column on test data, shape (n_features,)
    """
    # Input validation
    if X_train.shape[1] != X_test.shape[1]:
        raise ValueError("Training and test data must have same number of features")
    
    # Compute means from training data
    X_train_mean = X_train.mean(axis=0)
    y_train_mean = y_train.mean()
    
    # Center test data using training means
    X_test_centered = X_test - X_train_mean
    y_test_centered = y_test - y_train_mean
    
    # Compute coefficients using training data
    X_train_centered = X_train - X_train_mean
    y_train_centered = y_train - y_train_mean

    #beta = np.sum(X_train_centered * y_train_centered.reshape(-1, 1), axis=0) / np.sum(X_train_centered**2, axis=0)
    #AttributeError: 'Series' object has no attribute 'reshape'
    beta = np.sum(X_train_centered * y_train_centered.values.reshape(-1, 1), axis=0) / np.sum(X_train_centered**2, axis=0)

    # Compute predictions on test data
    y_pred = X_test_centered * beta + y_train_mean

    print("y_pred shape", y_pred.shape)
    print(y_pred)
    
    # Compute R² for each feature using test data
    r2_values = np.zeros(X_test.shape[1])
    for i in range(X_test.shape[1]):
        r2_values[i] = r2_score(y_test, y_pred[:, i])
    
    return r2_values



In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
data = pd.read_feather('/Users/matsumoton/Downloads/rat_data_starbase_maf01_M05_modeimpute.feather')
y=data['y']
data = data.drop(columns=['y'])
X_train, X_test, y_train, y_test = train_test_split(data, y, test_size=0.5, random_state=42)

In [7]:
# Get all encodings
all_encodings_train = map_snp_values(X_train, snp_pos=0, encoding_type='all', y=y_train)
all_encodings_test = map_snp_values(X_test, snp_pos=0, encoding_type='all', y=y_test)

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

def compute_r2_optimized(X_train, X_test, y_train, y_test):
    # Ensure inputs are numpy arrays
    X_train = np.asarray(X_train)
    X_test = np.asarray(X_test)
    y_train = np.asarray(y_train)
    y_test = np.asarray(y_test)

    # Preallocate array for R² scores
    r2_scores = np.zeros(X_train.shape[1])

    # Compute R² for each feature column
    for i in range(X_train.shape[1]):
        # Reshape for single feature regression
        X_train_col = X_train[:, i].reshape(-1, 1)
        X_test_col = X_test[:, i].reshape(-1, 1)

        # Fit linear regression model
        model = LinearRegression()
        model.fit(X_train_col, y_train)

        # Predict on test data
        y_pred = model.predict(X_test_col)

        # Compute R² score
        r2_scores[i] = r2_score(y_test, y_pred)

    return r2_scores

# Example usage
r2_values = compute_r2_optimized(all_encodings_train, all_encodings_test, y_train, y_test)
print("R^2 values for each encoding:")
for i, enc in enumerate(ENCODINGS):
    print(f"{enc}: {r2_values[i]:.16f}")

R^2 values for each encoding:
dominant: -0.0031609621219071
recessive: -0.0031609621219071
heterosis: -0.0022422672079965
underdominant: -0.0031450781350124
subadditive: -0.0031609621219071
superadditive: -0.0031609621219071
pager: -0.0030611864036856


In [37]:

import numpy as np

def compute_r2_matrix_aligned(X_train, X_test, y_train, y_test):
    # Ensure inputs are numpy arrays
    X_train = np.asarray(X_train)
    X_test = np.asarray(X_test)
    y_train = np.asarray(y_train)
    y_test = np.asarray(y_test)

    # Preallocate array for R² scores
    r2_scores = np.zeros(X_train.shape[1])

    # Loop over each column (feature)
    for i in range(X_train.shape[1]):
        # Extract individual feature column
        X_train_col = X_train[:, i]
        X_test_col = X_test[:, i]

        # Add bias term (intercept)
        X_train_bias = np.column_stack([X_train_col, np.ones(X_train_col.shape)])
        X_test_bias = np.column_stack([X_test_col, np.ones(X_test_col.shape)])

        # Compute weights using least squares
        weights = np.linalg.lstsq(X_train_bias, y_train, rcond=None)[0]

        # Predict test targets
        y_pred = X_test_bias @ weights

        # Compute R² for this feature
        ss_total = np.sum((y_test - np.mean(y_test)) ** 2)
        ss_residual = np.sum((y_test - y_pred) ** 2)
        r2_scores[i] = 1 - (ss_residual / ss_total)

    return r2_scores


# Example usage
r2_values = compute_r2_optimized(all_encodings_train, all_encodings_test, y_train, y_test)
print("R^2 values for each encoding:")
for i, enc in enumerate(ENCODINGS):
    print(f"{enc}: {r2_values[i]:.16f}")

R^2 values for each encoding:
dominant: -0.0031609621219071
recessive: -0.0031609621219071
heterosis: -0.0022422672079965
underdominant: -0.0031450781350124
subadditive: -0.0031609621219071
superadditive: -0.0031609621219071
pager: -0.0030611864036856


In [31]:
r2_values

np.float64(-0.002994376920724129)

In [9]:
# Get all encodings
all_encodings_train = map_snp_values(X_train, snp_pos=0, encoding_type='all', y=y_train)
all_encodings_test = map_snp_values(X_test, snp_pos=0, encoding_type='all', y=y_test)

# Compute R² values
r2_values = compute_r2_optimized(all_encodings_train, all_encodings_test, y_train, y_test)



y_pred shape (1583, 7)
[[ 0.0337955   0.0337955          nan ...  0.0337955   0.0337955
   0.0314427 ]
 [ 0.0337955   0.0337955          nan ...  0.0337955   0.0337955
   0.0314427 ]
 [ 0.0337955   0.0337955          nan ...  0.0337955   0.0337955
   0.0314427 ]
 ...
 [ 0.0337955   0.0337955          nan ...  0.0337955   0.0337955
   0.0314427 ]
 [ 0.0337955   0.0337955          nan ...  0.0337955   0.0337955
   0.0314427 ]
 [-0.04426199 -0.04426199         nan ... -0.04426199 -0.04426199
   0.0337955 ]]


/var/folders/t2/2rb77zbx4fvdt47gjll69t9h0000gt/T/ipykernel_49454/116236955.py:119: RuntimeWarning: invalid value encountered in divide
  beta = np.sum(X_train_centered * y_train_centered.values.reshape(-1, 1), axis=0) / np.sum(X_train_centered**2, axis=0)


ValueError: Input contains NaN.